In [1]:
import pandas as pd
import os
import numpy as np
import geopandas
import h3

In [2]:
os.getcwd()


'C:\\Users\\Fabian Weiland\\Documents\\GitHub\\urban-mobility-simulation\\simulation\\om_agent'

In [3]:
weekday_df = pd.read_csv(os.path.join("..", "..", "data", "demand", "weekday_hour_of_day_h3_start_8.csv"))

In [76]:
charging_allocation_ramin = ["871f18869ffffff","871f18b0dffffff","871f18b23ffffff","871f18b26ffffff","871f18b35ffffff","871f1d483ffffff","871f1d489ffffff","871f1d48bffffff","871f1d48effffff","871f1d49cffffff","871f1d4aaffffff","871f1d4b3ffffff","871f1d4d2ffffff","871f1d4d3ffffff","871f1d4f3ffffff","871f1d4f6ffffff"]

In [77]:
len(charging_allocation_ramin)

16

We map ramin's charging allocation in resolution 7 to resolution 8 (which maps to 7 hexagons each). Then, we select n individual cells based on demand information

In [78]:
weekday_df.head()

,is_holiday,hour_of_day,h3_start_8,h3_end_8,total_trips,sum_total_trips,od_multinom_p_est,o_poisson_est
0,False,0,881f188403fffff,881f188403fffff,0.0,3.0,0.0,0.018072
1,False,0,881f188403fffff,881f18841dfffff,0.0,3.0,0.0,0.018072
2,False,0,881f188403fffff,881f188691fffff,0.0,3.0,0.0,0.018072
3,False,0,881f188403fffff,881f188693fffff,0.0,3.0,0.0,0.018072
4,False,0,881f188403fffff,881f188695fffff,0.0,3.0,0.0,0.018072


In [4]:
demand_df = weekday_df[weekday_df["is_holiday"] == False].groupby("h3_start_8").agg({"sum_total_trips": "min"}).reset_index()

In [80]:
demand_df["h3_start_7"] = demand_df["h3_start_8"].apply(lambda x: h3.h3_to_parent(x))

In [81]:
demand_df = demand_df[demand_df["h3_start_7"].isin(charging_allocation_ramin)]

In [82]:
demand_df = demand_df.sort_values(by=["h3_start_7", "sum_total_trips"], ascending=False)

In [85]:
demand_df["use_station"] = ~demand_df["h3_start_7"].eq(demand_df["h3_start_7"].shift(2))

In [86]:
demand_df = demand_df[demand_df["use_station"]]

32 highest demand cells

In [19]:
import random

In [21]:
random.seed(42)

In [23]:
print(len(random.sample(np.array(demand_df.sort_values(by='sum_total_trips', ascending=False)["h3_start_8"]).tolist(), k=32)))

32


In [88]:
demand_df.groupby("h3_start_7").agg({"h3_start_8": "count"})

,h3_start_8
h3_start_7,
871f18869ffffff,2
871f18b0dffffff,2
871f18b23ffffff,2
871f18b26ffffff,2
871f18b35ffffff,2
871f1d483ffffff,2
871f1d489ffffff,2
871f1d48bffffff,2
871f1d48effffff,2


In [73]:
print([station for station in np.array(demand_df["h3_start_8"])])

['881f1d4f67fffff', '881f1d4f6dfffff', '881f1d4f3dfffff', '881f1d4f31fffff', '881f1d4d3dfffff', '881f1d4d31fffff', '881f1d4d29fffff', '881f1d4d2dfffff', '881f1d4b3dfffff', '881f1d4b39fffff', '881f1d4aa3fffff', '881f1d4aa9fffff', '881f1d49c9fffff', '881f1d49cbfffff', '881f1d48e9fffff', '881f1d48ebfffff', '881f1d48b3fffff', '881f1d48b1fffff', '881f1d4891fffff', '881f1d4899fffff', '881f1d4837fffff', '881f1d4831fffff', '881f18b35dfffff', '881f18b359fffff', '881f18b269fffff', '881f18b26bfffff', '881f18b23bfffff', '881f18b239fffff', '881f18b0dbfffff', '881f18b0d1fffff', '881f188691fffff', '881f188697fffff']


In [33]:
7 cellen pro ramin zelle
32/#ramin
--> 2 zellen pro ramin zelle auswählen

SyntaxError: invalid syntax (Temp/ipykernel_20864/1591230826.py, line 1)